In [1]:
import os, shutil
src = None
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    if "10_train_student_d32.py" in files:
        src = os.path.join(root, "10_train_student_d32.py"); break
assert src, "10_train_student_d32.py tidak ditemukan; tambahkan input rsna-knee-14-student-d32"
shutil.copy(src, "/kaggle/working/10_train_student_d32.py")
print("Script disalin dari:", src)

Script disalin dari: /kaggle/input/notebooks/kellenvalerie/rsna-knee-14-student-d32/10_train_student_d32.py


In [2]:
%%bash
cd /kaggle/working
COMMON="--alpha 0.5 --lr_backbone 1e-4 --oof oof_effb0_kf2.csv --backbone vit_small_patch14_dinov2.lvd142m --llrd 0.8"

# GPU 0: DINOv2 data v1 (3 view, 16 irisan), lalu DINOv2 data v2 seed 7
( CUDA_VISIBLE_DEVICES=0 python -u 10_train_student_d32.py $COMMON \
    --meta _meta.csv --batch 4 --epochs 12 --tag dinov2s_student_v1d \
    > log_dinov2s_student_v1d.txt 2>&1
  CUDA_VISIBLE_DEVICES=0 python -u 10_train_student_d32.py $COMMON \
    --meta _meta_v2.csv --batch 2 --epochs 12 --seed 7 --tag dinov2s_student_v3_s7 \
    > log_dinov2s_student_v3_s7.txt 2>&1 ) &

# GPU 1: DINOv2 32 irisan seed 7
CUDA_VISIBLE_DEVICES=1 python -u 10_train_student_d32.py $COMMON \
    --meta _meta_v3 --batch 1 --accum 2 --epochs 10 --seed 7 --tag dinov2s_student_d32_s7 \
    > log_dinov2s_student_d32_s7.txt 2>&1 &

wait
for t in dinov2s_student_v1d dinov2s_student_v3_s7 dinov2s_student_d32_s7; do
  echo "===== $t ====="; tail -n 16 log_$t.txt
done
grep -H "Volume :\|Label  :\|Label diperbaiki\|Jumlah irisan\|Jumlah view\|LLRD\|Train " log_*.txt

===== dinov2s_student_v1d =====
Epoch 11: loss 0.3742 | val AUC 0.8753 | val-ref AUC 0.8903 | gold AUC 0.8610 | 13.8 menit

=== [dinov2s_student_v1d] Checkpoint terbaik (val-ref AUC 0.8903) pada label emas ===
ACL                AUC=0.907
MCL                AUC=0.880
Medial Meniscus    AUC=0.892
Lateral Meniscus   AUC=0.764
Medial OA          AUC=0.975
Lateral OA         AUC=0.787
PF OA              AUC=0.783
Effusion           AUC=0.911
Synovitis          AUC=0.710
Baker's            AUC=0.975
Contusion          AUC=0.873
Fracture           AUC=0.876
Mean AUC gold: 0.8610
===== dinov2s_student_v3_s7 =====
Epoch 11: loss 0.3809 | val AUC 0.8725 | val-ref AUC 0.8888 | gold AUC 0.8483 | 23.3 menit

=== [dinov2s_student_v3_s7] Checkpoint terbaik (val-ref AUC 0.8888) pada label emas ===
ACL                AUC=0.863
MCL                AUC=0.819
Medial Meniscus    AUC=0.879
Lateral Meniscus   AUC=0.747
Medial OA          AUC=0.971
Lateral OA         AUC=0.799
PF OA              AUC=0.798
Eff

In [3]:
import pandas as pd
for tag in ["dinov2s_student_v1d", "dinov2s_student_v3_s7", "dinov2s_student_d32_s7"]:
    print(f"\n===== {tag} =====")
    try:
        print(pd.read_csv(f"/kaggle/working/train_log_{tag}.csv").round(4).to_string(index=False))
    except FileNotFoundError:
        print("log belum ada")


===== dinov2s_student_v1d =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5434   0.7459       0.7570    0.7588 13.8161
     1 0.4738   0.8099       0.8258    0.8359 13.7595
     2 0.4503   0.8198       0.8373    0.8281 13.7828
     3 0.4372   0.8419       0.8607    0.8340 13.7706
     4 0.4261   0.8473       0.8656    0.8383 13.7827
     5 0.4162   0.8596       0.8753    0.8529 13.8305
     6 0.4061   0.8650       0.8817    0.8639 13.8833
     7 0.3966   0.8721       0.8872    0.8654 14.0177
     8 0.3880   0.8722       0.8872    0.8608 13.9691
     9 0.3810   0.8732       0.8882    0.8599 13.8385
    10 0.3768   0.8752       0.8902    0.8597 13.8489
    11 0.3742   0.8753       0.8903    0.8610 13.8155

===== dinov2s_student_v3_s7 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5455   0.7314       0.7453    0.7349 23.2349
     1 0.4912   0.7914       0.8095    0.7784 23.3092
     2 0.4648   0.8122       0.8286    0.8020 23.2757
     3 0.4495 